# Measure time-evolved expectation values in `qdk-chemistry`

## This notebook demonstrates the `EvolveAndMeasure` algorithm for Hamiltonian time evolution and observable measurement using `qdk-chemistry`. It shows how to:

1. Define a driven qubit Hamiltonian $H(t) = H_0 + f(t)\,H_1$
2. Build a Trotter evolution circuit
3. Run the simulation **without noise** using the QDK full-state simulator
4. Run the simulation **with noise** using both the QDK and Qiskit Aer backends
5. Optionally transpile to a target basis gate set before execution

In [ ]:
import numpy as np

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef, DrivenQubitHamiltonian, LatticeGraph, QuantumErrorProfile,
    QubitHamiltonian,
)
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)


We define a **driven qubit Hamiltonian** $H(t) = H_0 + f(t)\,H_1$ where the static part $H_0$ contains ZZ couplings and the driven part $H_1$ is a transverse field modulated by a smooth drive function $f(t)$. The observable is $ZZ$, measured after the full evolution sequence.

`DrivenQubitHamiltonian` models a continuously driven Hamiltonian:

$$H(t) = H_0 + f(t)\,H_1$$

where:
- $H_0 = J\,\sigma_z^{(0)}\sigma_z^{(1)}$ is the static Ising ZZ coupling
- $H_1 = h\,(\sigma_x^{(0)} + \sigma_x^{(1)})$ is the transverse field
- $f(t) = \sin(\omega\,t)$ is a smooth sinusoidal drive

In [ ]:
lattice = LatticeGraph.chain(2)

# Static part: ZZ coupling only
h0 = create_ising_hamiltonian(lattice, j=1.0, h=0.0)

# Driven part: transverse field only
h1 = create_ising_hamiltonian(lattice, j=0.0, h=0.5)

# Smooth sinusoidal drive
omega = 2 * np.pi  # one full cycle per unit time
total_time = 2.0

drive = lambda t: np.sin(omega * t)

td_hamiltonian = DrivenQubitHamiltonian(h0, h1, drive=drive)

observable = QubitHamiltonian(["ZZ"], np.array([1.0]))

# Identity state prep (|00⟩ initial state)
from qdk_chemistry.data import Circuit
from qdk_chemistry.data.circuit import QsharpFactoryData
from qdk_chemistry.utils.qsharp import QSHARP_UTILS

params = {"pauliExponents": [], "pauliCoefficients": [], "repetitions": 1}
targets = list(range(td_hamiltonian.num_qubits))
state_prep = Circuit(
    qsharp_op=QSHARP_UTILS.PauliExp.MakeRepPauliExpOp(params),
    qsharp_factory=QsharpFactoryData(
        program=QSHARP_UTILS.PauliExp.MakeRepPauliExpCircuit,
        parameter={"evo_params": params, "target_indices": targets},
    ),
)


## Exact evolution via matrix exponential

Compute the exact time evolution $|\psi(T)\rangle$ by dividing $[0, T]$ into fine steps and applying $e^{-i\,H(t_k)\,\delta t}$ at each step. This serves as the reference value for the Trotter-based simulation.

In [ ]:
from scipy.sparse.linalg import expm_multiply

# Convert H0, H1 to sparse matrices
H0_sparse = h0.to_matrix(sparse=True)
H1_sparse = h1.to_matrix(sparse=True)

num_qubits = h0.num_qubits

# Initial state |00...0⟩
psi = np.zeros(2**num_qubits, dtype=complex)
psi[0] = 1.0

# Fine time-stepping for the exact reference
n_exact_steps = 2000
dt_exact = total_time / n_exact_steps

for k in range(n_exact_steps):
    t_mid = (k + 0.5) * dt_exact
    H_t = H0_sparse + drive(t_mid) * H1_sparse
    psi = expm_multiply(-1j * H_t * dt_exact, psi)

# Compute exact ⟨ZZ⟩
Z = np.array([1, -1], dtype=complex)
ZZ_diag = np.kron(Z, Z)
zz_exact = np.real(np.conj(psi) @ (ZZ_diag * psi))

print(f"Exact ⟨ZZ⟩: {zz_exact:.6f}")


## Noiseless simulation (QDK full-state simulator)
Set up the `measure_simulation` algorithm and configure its nested algorithms via settings.

In [ ]:
measure_simulation = create("measure_simulation", "evolve_and_measure")
measure_simulation.settings().set(
    "evolution_builder",
    AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=20, order=1),
)
measure_simulation.settings().set("total_time", total_time)


Run the evolution and measure `⟨ZZ⟩` without any noise using the QDK full-state simulator.

In [ ]:
measure_simulation.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_full_state_simulator"),
)
measure_simulation.settings().set("shots", 10000)

measurements_noiseless = measure_simulation.run(
    td_hamiltonian,
    observables=[observable],
    state_prep=state_prep,
)

zz_noiseless = measurements_noiseless[0][0].energy_expectation_value
print(f"Noiseless ⟨ZZ⟩: {zz_noiseless:.6f}")


## Define a noise profile

`QuantumErrorProfile` provides a backend-agnostic way to specify depolarizing noise on individual gates. This profile can be used with both the QDK and Qiskit Aer simulators.

> **Note:** The QDK full-state simulator applies noise at the *native gate level* of the compiled QIR. If the circuit uses high-level Pauli exponentials (e.g. via `PauliSequenceMapper`), those are executed natively without decomposition into `cx`/`rz` gates — so noise on `cx` won't apply. To see noise with the QDK simulator, either use `basis_gates` to force decomposition, or define noise on the native gates (`rzz`, `rx`, etc.).

In [ ]:
qdk_error_profile = QuantumErrorProfile(
    name="demo_noise",
    description="Light depolarizing noise on common gates",
    errors={
        "cx":  {"depolarizing_error": 0.01},
        "cz":  {"depolarizing_error": 0.01},
        "rzz": {"depolarizing_error": 0.01},
        "h":   {"depolarizing_error": 0.001},
        "rz":  {"depolarizing_error": 0.001},
        "rx":  {"depolarizing_error": 0.001},
        "s":   {"depolarizing_error": 0.001},
        "sdg": {"depolarizing_error": 0.001},
    },
)

## Noisy simulation — QDK full-state simulator

Because `PauliSequenceMapper` produces native Pauli-exponential operations, and our noise profile includes `rzz` (the native two-qubit gate), the QDK simulator **will** apply noise to those operations.

In [ ]:
measurements_noisy_qdk = measure_simulation.run(
    td_hamiltonian,
    observables=[observable],
    state_prep=state_prep,
    noise=qdk_error_profile,
)

zz_noisy_qdk = measurements_noisy_qdk[0][0].energy_expectation_value
print(f"Noisy QDK ⟨ZZ⟩: {zz_noisy_qdk:.6f}")


## Noisy simulation — Qiskit Aer simulator

The Qiskit Aer backend transpiles the circuit to primitive gates (`cx`, `rz`, `h`, …) before execution, so noise on those gates fires naturally.

In [ ]:
measure_simulation_aer = create("measure_simulation", "evolve_and_measure")
measure_simulation_aer.settings().set(
    "evolution_builder",
    AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=20, order=1),
)
measure_simulation_aer.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qiskit_aer_simulator"),
)
measure_simulation_aer.settings().set("shots", 10000)
measure_simulation_aer.settings().set("total_time", total_time)

measurements_noisy_aer = measure_simulation_aer.run(
    td_hamiltonian,
    observables=[observable],
    state_prep=state_prep,
    noise=qdk_error_profile,
)

zz_noisy_aer = measurements_noisy_aer[0][0].energy_expectation_value
print(f"Noisy Aer ⟨ZZ⟩: {zz_noisy_aer:.6f}")


## Compare results

Side-by-side comparison of the noiseless and noisy expectation values.

In [ ]:
print("Simulator               ⟨ZZ⟩")
print("─" * 36)
print(f"Exact                 {zz_exact: .6f}")
print(f"QDK  (noiseless)      {zz_noiseless: .6f}")
print(f"QDK  (noisy)          {zz_noisy_qdk: .6f}")
print(f"Aer  (noisy)          {zz_noisy_aer: .6f}")